# Cocktail Engine Build (SQL + Python)
  ### Goal

#### Build a cocktail analytics and forecasting system that models sales demand, event-driven uplifts, and financial performance for a hotel cocktail bar.
        
#### The system combines SQL data modelling, Python analytics, and financial calculations to produce a 24-month forecast of cocktail sales, revenue, and gross profit.
        
#### The project demonstrates how operational bar data can be transformed into a commercial forecasting engine used to evaluate menu initiatives and supplier agreements.



### Imports + DB path

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../db/cocktail_engine.db")
DB_PATH


PosixPath('../db/cocktail_engine.db')

In [2]:
import sqlite3, pandas as pd

# DB_PATH must already be defined above
with sqlite3.connect(DB_PATH) as con:
    # Detect price column in fact_cocktail_price
    price_cols = pd.read_sql("PRAGMA table_info('fact_cocktail_price');", con)["name"].tolist()
    price_candidates = ["sell_price", "price_per_unit", "price", "menu_price", "unit_price"]
    PRICE_COL = next((c for c in price_candidates if c in price_cols), None)
    if PRICE_COL is None:
        raise ValueError(f"No usable price column found in fact_cocktail_price. Columns={price_cols}")

    # Detect cost column in v_cost_per_cocktail
    cost_cols = pd.read_sql("PRAGMA table_info('v_cost_per_cocktail');", con)["name"].tolist()
    cost_candidates = ["cost_per_cocktail", "cost_per_unit"]
    COST_COL = next((c for c in cost_candidates if c in cost_cols), None)
    if COST_COL is None:
        raise ValueError(f"No usable cost column found in v_cost_per_cocktail. Columns={cost_cols}")

    con.execute("DROP VIEW IF EXISTS v_reality_financial_14g;")
    con.execute(f"""
        CREATE VIEW v_reality_financial_14g AS
        SELECT
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,
            f.baseline_ma3,
            f.uplift,
            f.reality_forecast_qty AS forecast_qty,
            p.{PRICE_COL}          AS sell_price,
            c.{COST_COL}           AS cost_per_cocktail,
            ROUND(f.reality_forecast_qty * p.{PRICE_COL}, 2)                    AS forecast_revenue,
            ROUND(f.reality_forecast_qty * c.{COST_COL}, 2)                     AS forecast_cogs,
            ROUND((f.reality_forecast_qty * p.{PRICE_COL}) -
                  (f.reality_forecast_qty * c.{COST_COL}), 2)                  AS forecast_gross_profit
        FROM fact_reality_forecast_14f f
        LEFT JOIN fact_cocktail_price p
          ON p.cocktail_id = f.cocktail_id
        LEFT JOIN v_cost_per_cocktail c
          ON c.cocktail_id = f.cocktail_id;
    """)

print(f"✅ Rebuilt v_reality_financial_14g using price={PRICE_COL}, cost={COST_COL}")

✅ Rebuilt v_reality_financial_14g using price=sell_price, cost=cost_per_cocktail


## 1) Create the Tables

In [3]:
con = sqlite3.connect(DB_PATH)

con.executescript("""
PRAGMA foreign_keys = OFF;

DROP TABLE IF EXISTS bridge_recipe;
DROP TABLE IF EXISTS fact_cocktail_price;
DROP TABLE IF EXISTS dim_ingredient;
DROP TABLE IF EXISTS dim_cocktail;

PRAGMA foreign_keys = ON;

-- DIMENSION: Ingredients (one row per ingredient)
DROP TABLE IF EXISTS dim_ingredient;
CREATE TABLE dim_ingredient (
    ingredient_id INTEGER PRIMARY KEY AUTOINCREMENT,  -- unique ID, auto assigned
    ingredient_name TEXT UNIQUE NOT NULL,              -- human label, must be unique and present
    bottle_ml REAL NOT NULL,                           -- decimals allowed
    bottle_price REAL NOT NULL,
    cost_per_ml REAL NOT NULL                          -- derived = bottle_price / bottle_ml
);

-- DIMENSION: Cocktails (one row per cocktail)
DROP TABLE IF EXISTS dim_cocktail;
CREATE TABLE dim_cocktail (
    cocktail_id INTEGER PRIMARY KEY AUTOINCREMENT,
    cocktail_name TEXT UNIQUE NOT NULL
);

-- BRIDGE: Recipes (many-to-many between cocktails and ingredients)
DROP TABLE IF EXISTS bridge_recipe;
CREATE TABLE bridge_recipe (
    cocktail_id INTEGER NOT NULL,          -- must point to a real cocktail
    ingredient_id INTEGER NOT NULL,        -- must point to a real ingredient
    ml_per_serve REAL NOT NULL,            -- the “amount” for this recipe line
    include_in_cost INTEGER NOT NULL DEFAULT 1,  -- 1 include, 0 exclude (juice/champagne/etc.)
    notes TEXT,

    -- prevents duplicate recipe lines for same ingredient in same cocktail
    PRIMARY KEY (cocktail_id, ingredient_id),

    -- relationship rules:
    FOREIGN KEY (cocktail_id) REFERENCES dim_cocktail(cocktail_id),
    FOREIGN KEY (ingredient_id) REFERENCES dim_ingredient(ingredient_id)
);

-- FACT: Sell price (separate because price changes independently)
DROP TABLE IF EXISTS fact_cocktail_price;
CREATE TABLE fact_cocktail_price (
    cocktail_id INTEGER PRIMARY KEY,       -- one price per cocktail (for now)
    sell_price REAL NOT NULL,
    FOREIGN KEY (cocktail_id) REFERENCES dim_cocktail(cocktail_id)
);
""")

con.commit()


print("✅ Schema created")


✅ Schema created


In [4]:
con = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", con)

,name
0,bridge_recipe
1,config_active_scenario
2,dim_cocktail
3,dim_demand_sensitivity_11a
4,dim_inflation_scenario_9a
5,dim_ingredient
6,dim_month_14a
7,dim_scenario
8,dim_scenario_10a
9,fact_2026_baseline_7d


## 2) Insert ingredients (starter set) + helper functions

In [5]:
def upsert_ingredient(name, bottle_price, bottle_ml):
    
    cost_per_ml = bottle_price / bottle_ml

    con.execute("""
        INSERT INTO dim_ingredient (ingredient_name, bottle_ml, bottle_price, cost_per_ml)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(ingredient_name) DO UPDATE SET
            bottle_ml=excluded.bottle_ml,
            bottle_price=excluded.bottle_price,
            cost_per_ml=excluded.cost_per_ml;
    """, (name, bottle_ml, bottle_price, cost_per_ml))

def upsert_cocktail(name):
    con.execute("""
        INSERT INTO dim_cocktail (cocktail_name)
        VALUES (?)
        ON CONFLICT(cocktail_name) DO NOTHING;
    """, (name,))

def get_id(table, id_col, name_col, value):
    row = con.execute(
        f"SELECT {id_col} FROM {table} WHERE {name_col} = ?",
        (value,)
    ).fetchone()
    if not row:
        raise ValueError(f"Missing in {table}: {value}")
    return row[0]

def upsert_price(cocktail_name, sell_price):
    upsert_cocktail(cocktail_name)
    cid = get_id("dim_cocktail", "cocktail_id", "cocktail_name", cocktail_name)
    con.execute("""
        INSERT INTO fact_cocktail_price (cocktail_id, sell_price)
        VALUES (?, ?)
        ON CONFLICT(cocktail_id) DO UPDATE SET
            sell_price=excluded.sell_price;
    """, (cid, float(sell_price)))

def upsert_recipe_line(cocktail_name, ingredient_name, ml_per_serve, include_in_cost=1, notes=None):
    upsert_cocktail(cocktail_name)
    cid = get_id("dim_cocktail", "cocktail_id", "cocktail_name", cocktail_name)
    iid = get_id("dim_ingredient", "ingredient_id", "ingredient_name", ingredient_name)

    con.execute("""
        INSERT INTO bridge_recipe (cocktail_id, ingredient_id, ml_per_serve, include_in_cost, notes)
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT(cocktail_id, ingredient_id) DO UPDATE SET
            ml_per_serve=excluded.ml_per_serve,
            include_in_cost=excluded.include_in_cost,
            notes=excluded.notes;
    """, (cid, iid, float(ml_per_serve), int(include_in_cost), notes))

con.execute("PRAGMA foreign_keys = ON;")
con.commit()

In [6]:
import sqlite3
import pandas as pd

def norm(s: str) -> str:
    return " ".join(s.strip().split())

def upsert_ingredient_price(con, name, bottle_price, bottle_ml):
    name = norm(name)
    bottle_ml = float(bottle_ml)
    bottle_price = float(bottle_price)
    cost_per_ml = bottle_price / bottle_ml

    con.execute("""
        INSERT INTO dim_ingredient (ingredient_name, bottle_ml, bottle_price, cost_per_ml)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(ingredient_name) DO UPDATE SET
            bottle_ml=excluded.bottle_ml,
            bottle_price=excluded.bottle_price,
            cost_per_ml=excluded.cost_per_ml;
    """, (name, bottle_ml, bottle_price, cost_per_ml))

# ---- PRICES ----
spirit_prices = [
    ("Plymouth Gin", 24.99, 700),
    ("Absolut Elyx", 34.99, 700),
    ("Del Maguey Vida Mezcal", 42.00, 750),
    ("Whistle Pig", 85.75, 700),
    ("St Germain", 32.75, 700),
    ("Grey Goose Vodka", 43.95, 700),
    ("Yellow Chartreuse", 47.75, 700),
    ("Cocchi Torino", 27.50, 700),
    ("Campari", 21.50, 700),
    ("Cointreau", 27.00, 700),
    ("Havana 7", 28.50, 700),
    ("Hennessy", 41.95, 700),
    ("Italicus", 32.50, 700),
    ("Rabbit Hole Whiskey Infused Coffee", 47.25, 700),
    ("Chivas 12y", 24.75, 700),
    ("Seedlip Garden", 22.50, 700),
    ("Everleaf Forest", 21.50, 700),
    ("Acqua di Cedro", 40.50, 700),
    ("Midori", 18.95, 700),
    ("Ancho Reyes", 43.95, 700),
    ("Tia Maria", 16.95, 700),
    ("Disaronno", 25.50, 700),
    ("Amaro Montenegro", 24.50, 700),
    ("Dark Cacao Liqueur", 27.25, 700),
    ("Chambord", 23.75, 700),
    ("Lychee Liqueur", 22.50, 700),
    ("Parafante Fig Liqueur", 22.75, 500),
    ("Mancino Sakura Vermouth", 27.95, 500),
    ("La Hechicera Reserva", 44.95, 700),
    ("Akashi Tai Sake", 24.00, 720),
    ("Orange Bitters", 19.75, 1000),
    ("Monkey 47", 50.00, 700),
    ("Chinotto Muyu", 34.25, 720)
]

house_made = [
    "Yuzu & Ginger Syrup", "Earl Grey Syrup", "Eucalyptus Syrup",
    "Garden Mint", "Sugar Syrup", "Vanilla Syrup",
    "Matcha Syrup", "Sherry Syrup", "Homemade Sweetcorn Syrup",
    "Apricot Syrup", "Green Apple Sherbet", "Cucumber Sherbet",
    "Jasmin Green Pearls Cordial", "Rose Wine Syrup"
]

juice_prices = [
    ("Lemon Juice", 2.40, 1000),
    ("Lime Juice", 2.40, 1000),
    ("Apple Juice", 1.80, 1000),
    ("Grapefruit Juice", 2.00, 1000),
    ("Passion Fruit Puree", 4.50, 1000),
    ("Pineapple Juice", 3.80, 1000),
]

champagne = [
    ("Ayala Champagne", 74.99, 750),
]

# ---- APPLY ----
with sqlite3.connect(DB_PATH) as con:
    for name, price, ml in spirit_prices:
        upsert_ingredient_price(con, name, price, ml)

    for name in house_made:
        upsert_ingredient_price(con, name, 0.50, 700)

    for name, price, ml in juice_prices:
        upsert_ingredient_price(con, name, price, ml)

    for name, price, ml in champagne:
        upsert_ingredient_price(con, name, price, ml)

print("✅ Pricing applied (committed)")

# ---- VERIFY (separate read; cannot affect commit) ----
with sqlite3.connect(DB_PATH) as con:
    df = pd. read_sql("""
        SELECT ingredient_name, bottle_ml, bottle_price, ROUND(cost_per_ml, 5) AS cost_per_ml
        FROM dim_ingredient
        ORDER BY bottle_price DESC
        LIMIT 25;
    """, con)
con.commit()
df

✅ Pricing applied (committed)


,ingredient_name,bottle_ml,bottle_price,cost_per_ml
0,Whistle Pig,700.0,85.75,0.12250
1,Ayala Champagne,750.0,74.99,0.09999
2,Monkey 47,700.0,50.00,0.07143
3,Yellow Chartreuse,700.0,47.75,0.06821
4,Rabbit Hole Whiskey Infused Coffee,700.0,47.25,0.06750
5,La Hechicera Reserva,700.0,44.95,0.06421
6,Grey Goose Vodka,700.0,43.95,0.06279
7,Ancho Reyes,700.0,43.95,0.06279
8,Del Maguey Vida Mezcal,750.0,42.00,0.05600
9,Hennessy,700.0,41.95,0.05993


#### Bottle size updates

In [7]:
import sqlite3
import pandas as pd


# --- 250ml bitters ---
bottle_250 = [
    "Orange Bitters",
]

# --- 500ml bottles ---
bottle_500 = [
    "Mancino Sakura Vermouth", 
    "Monkey 47",
    "Parafante Fig Liqueur",
    "Chinotto Muyu"
    
]

# --- 720ml bottles ---
bottle_720 = [
    "Akashi Tai Sake"
]
# --- 750ml bottles ---
bottle_750 = [
    "Del Maguey Vida Mezcal",
]

# --- 1000ml bottles ---
bottle_1000 = [
    "Apple Juice",
    "Grapefruit Juice",
    "Lemon Juice",
    "Lime Juice",
    "Grapefruit Juice"
]

# --- 1500ml bottles ---
bottle_1500 = [
    "Ayala Champagne",

]

def set_bottle_ml(name, bottle_ml):
    # keep price as-is; only update bottle_ml, and recompute cost_per_ml safely
    row = con.execute("""
        SELECT bottle_price
        FROM dim_ingredient
        WHERE ingredient_name = ?
    """, (name,)).fetchone()

    if row is None:
        raise ValueError(f"Ingredient not found: {name}")

    bottle_price = float(row[0])
    cost_per_ml = bottle_price / float(bottle_ml) if bottle_ml else 0

    con.execute("""
        UPDATE dim_ingredient
        SET bottle_ml = ?, cost_per_ml = ?
        WHERE ingredient_name = ?;
    """, (float(bottle_ml), float(cost_per_ml), name))

for n in bottle_250:
    set_bottle_ml(n, 250)

for n in bottle_500:
    set_bottle_ml(n, 500)

for n in bottle_720:
    set_bottle_ml(n, 720)

for n in bottle_750:
    set_bottle_ml(n, 750)

for n in bottle_1000:
    set_bottle_ml(n, 1000)
    
for n in bottle_1500:
    set_bottle_ml(n, 1500)

con.commit()

pd.read_sql(""" 
    SELECT ingredient_name, bottle_ml, bottle_price, cost_per_ml
    FROM dim_ingredient
    WHERE ingredient_name IN (
                'Orange Bitters', 
                'Monkey 47', 
                'Del Maguey Vida Mezcal', 
                'Ayala Champagne', 
                'Mancino Sakura Vermouth', 
                'Akashi Tai Sake', 
                "Fig Liqueur", 
                "Apple Juice",
                "Grapefruit Juice",
                "Lemon Juice",
                "Lime Juice",
                "Grapefruit Juice",
                "Parafante Fig Liqueur",
                "Chinotto Muyu" 
                )
    ORDER BY ingredient_name;
    """, con)


,ingredient_name,bottle_ml,bottle_price,cost_per_ml
0,Akashi Tai Sake,720.0,24.00,0.033333
1,Apple Juice,1000.0,1.80,0.001800
2,Ayala Champagne,1500.0,74.99,0.049993
3,Chinotto Muyu,500.0,34.25,0.068500
4,Del Maguey Vida Mezcal,750.0,42.00,0.056000
5,Grapefruit Juice,1000.0,2.00,0.002000
6,Lemon Juice,1000.0,2.40,0.002400
7,Lime Juice,1000.0,2.40,0.002400
8,Mancino Sakura Vermouth,500.0,27.95,0.055900
9,Monkey 47,500.0,50.00,0.100000


## 3) Insert cocktail names

In [8]:
cocktails = [
    "Ginger Martini","Earl Grey Martini","Tipsy Koala","Celeste","Cool Calm Cucumber",
    "Hot Stuff","Garden of Temptation","Daylight Robbery","The Devil’s Tea","Pink Flamingo",
    "The Smooch","Ship’s Rumbullion","BLT","Ristretto Martini","Floral Twist","Green Dream",
    "Fruity Old Devil","Garden Negroni","Churchill Manhattan","TYYM","George Martini",
    "Ohara Punch","Deep Sea Cosmo","Mr Sweeney","Negroni","French Martini", "Negroni"
]

for c in cocktails:
    upsert_cocktail(c)

con.commit()
pd.read_sql("SELECT * FROM dim_cocktail ORDER BY cocktail_name;", con)

,cocktail_id,cocktail_name
0,13,BLT
1,4,Celeste
2,19,Churchill Manhattan
3,5,Cool Calm Cucumber
4,8,Daylight Robbery
5,23,Deep Sea Cosmo
6,2,Earl Grey Martini
7,15,Floral Twist
8,26,French Martini
9,17,Fruity Old Devil


#### Insert ingredient names (placeholder bottle_price=0)


In [9]:
import sqlite3
import pandas as pd

def clean_name(x: str) -> str:
    return " ".join(x.strip().split())

def upsert_ingredient_placeholder(con, name, bottle_ml=700):
    name = clean_name(name)
    con.execute("""
        INSERT INTO dim_ingredient (ingredient_name, bottle_ml, bottle_price, cost_per_ml)
        VALUES (?, ?, 0.0, 0.0)
        ON CONFLICT(ingredient_name) DO UPDATE SET
            bottle_ml=excluded.bottle_ml;
    """, (name, float(bottle_ml)))


ingredients = [
    # Ginger Martini
    "Plymouth Gin", "Akashi Tai Sake", "Yuzu & Ginger Syrup", "Orange Bitters",

    # Earl Grey Martini
    "Absolut Elyx", "Earl Grey Syrup", "Mancino Sakura Vermouth",

    # Tipsy Koala
    "Malfy Gin Rosa", "Italicus", "Acqua di Cedro", "Grapefruit Juice", "Eucalyptus Syrup", "Garden Mint",

    # Celeste
    "Hennessy", "Chinotto Muyu", "Parafante Fig Liqueur", "Sugar Syrup", "Lemon Juice", "Ayala Champagne",

    # Cool Calm Cucumber
    "Monkey 47", "Midori", "Lychee Liqueur", "Cucumber Sherbet", "Miraculous Foam", "Garden Mint",

    # Hot Stuff
    "Del Maguey Vida Mezcal", "Ancho Reyes", "Vanilla Syrup","Lime Juice", "Champagne Foam", 

    # Garden of Temptation
    "Grey Goose Vodka", "Apple Juice", "Manzana Verde Liqueur", "Matcha Syrup",

    # Daylight Robbery
    "Altos Plata", "Cointreau", "Absinthe Wash"
]

more_ingredients = [
    # Devil's Tea
    "La Tomato", "Humami Vermouth", "Cocchi Rosa", "Wine Sherbet",

    # Pink Flamingo
    "Chambord", "Figue Liqueur", "Jasmin Green Pearls Cordial",

    # The Smooch
    "La Hechichera Reserva", "Kahlua", "Dark Cacao Liqueur", "Baileys",

    # Ship’s Rumbullion
    "Havana 7yr", "Tia Maria", "Disaronno", "Sherry Syrup", "Lime Juice", 

    # BLT
    "Whistle Pig", "Crème de Banana", "Homemade Sweetcorn Syrup", "Mancino Dry Vermouth",

    # Ristretto Martini
    "Rabbit Hole Whiskey Infused Coffee", "Amaro Montenegro", "Cocchi Torino",

    # NA
    "Peppermint Cordial", "Everleaf Forest", "Elderflower Cordial", "Orgeat",
    "Supasawa", "Seedlip Garden", "Apricot Syrup", "Opius Albedo",
    "Green Apple Sherbet", "Sicilian Lemon Tonic",

    # Classics
    "Yellow Chartreuse", "Chivas 12y", "Cacao Blanc Briottet",
    "Karminia Barolo Chinato", "Goring Gin", "St Germain",
    "Crème de Peche", "Sherbet", "Rose Wine Syrup",

    # Others
    "Absolut Citron", "Limoncello", "Havana 7",
    "Passion Fruit Puree", "Rum Caviar",
    "Broken Clock Vodka", "Yuzu Orange", "Dom Benedictine",
    "Maraschino Cherries",

    # Negroni
    " Plymouth Gin", "Campari", "Mancino Rosso"
]

all_names = sorted(set(map(clean_name, ingredients + more_ingredients)))

with sqlite3.connect(DB_PATH) as con:
    for name in all_names:
        upsert_ingredient_placeholder(con, name, bottle_ml=700)

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT ingredient_id, ingredient_name, bottle_ml, bottle_price, cost_per_ml
        FROM dim_ingredient
        ORDER BY ingredient_name;
    """, con)

df

,ingredient_id,ingredient_name,bottle_ml,bottle_price,cost_per_ml
0,55,Absinthe Wash,700.0,0.00,0.000000
1,56,Absolut Citron,700.0,0.00,0.000000
2,2,Absolut Elyx,700.0,34.99,0.049986
3,18,Acqua di Cedro,700.0,40.50,0.057857
4,30,Akashi Tai Sake,700.0,24.00,0.033333
...,...,...,...,...,...
85,4,Whistle Pig,700.0,85.75,0.122500
86,139,Wine Sherbet,700.0,0.00,0.000000
87,7,Yellow Chartreuse,700.0,47.75,0.068214
88,34,Yuzu & Ginger Syrup,700.0,0.50,0.000714


## 4) Insert recipes

In [10]:
# Ginger Martini
upsert_recipe_line("Ginger Martini", "Plymouth Gin", 40, 1)
upsert_recipe_line("Ginger Martini", "Akashi Tai Sake", 20, 1)
upsert_recipe_line("Ginger Martini", "Yuzu & Ginger Syrup", 10, 1)
upsert_recipe_line("Ginger Martini", "Orange Bitters", 2, 1, notes="dash treated as ml")

# Earl Grey Martini
upsert_recipe_line("Earl Grey Martini", "Absolut Elyx", 50, 1)
upsert_recipe_line("Earl Grey Martini", "Earl Grey Syrup", 10, 1)
upsert_recipe_line("Earl Grey Martini", "Mancino Sakura Vermouth", 10, 1)

# Tipsy Koala
upsert_recipe_line("Tipsy Koala", "Malfy Gin Rosa", 30, 1)
upsert_recipe_line("Tipsy Koala", "Italicus", 10, 1)
upsert_recipe_line("Tipsy Koala", "Acqua di Cedro", 15, 1)
upsert_recipe_line("Tipsy Koala", "Grapefruit Juice", 25, 0, notes="fresh")
upsert_recipe_line("Tipsy Koala", "Eucalyptus Syrup", 5, 1)
upsert_recipe_line("Tipsy Koala", "Garden Mint", 1, 1)

# --- Celeste
upsert_recipe_line("Celeste", "Hennessy", 40, 1)
upsert_recipe_line("Celeste", "Chinotto Muyu", 20, 1)
upsert_recipe_line("Celeste", "Parafante Fig Liqueur", 15, 1)
upsert_recipe_line("Celeste", "Sugar Syrup", 10, 1)
upsert_recipe_line("Celeste", "Lemon Juice", 5, 0, notes="fresh")
upsert_recipe_line("Celeste", "Ayala Champagne", 20, 0, notes="top")


# Remaninder of list insert 

# --- BLT ---
upsert_recipe_line("BLT", "Whistle Pig", 50, 1)
upsert_recipe_line("BLT", "Crème de Banana", 5, 1)
upsert_recipe_line("BLT", "Homemade Sweetcorn Syrup", 5, 1)
upsert_recipe_line("BLT", "Mancino Dry Vermouth", 10, 1)
upsert_recipe_line("BLT", "Absinthe Wash", 0, 0)

# --- Ristretto Martini ---
upsert_recipe_line("Ristretto Martini", "Rabbit Hole Whiskey Infused Coffee", 25, 1)
upsert_recipe_line("Ristretto Martini", "Amaro Montenegro", 25, 1)
upsert_recipe_line("Ristretto Martini", "Cocchi Torino", 25, 1)
upsert_recipe_line("Ristretto Martini", "Orange Bitters", 2, 1, notes= 'Dash treated as ml')

# --- Floral Twist ---
upsert_recipe_line("Floral Twist", "Peppermint Cordial", 35, 1)
upsert_recipe_line("Floral Twist", "Everleaf Forest", 15, 1)
upsert_recipe_line("Floral Twist", "Elderflower Cordial", 15, 1)
upsert_recipe_line("Floral Twist", "Orgeat", 5, 1)
upsert_recipe_line("Floral Twist", "Supasawa", 0, 0)

# --- Green Dream ---
upsert_recipe_line("Green Dream", "Seedlip Garden", 50, 1)
upsert_recipe_line("Green Dream", "Apple Juice", 30, 0, notes= 'Fresh')
upsert_recipe_line("Green Dream", "Apricot Syrup", 15, 1)
upsert_recipe_line("Green Dream", "Lemon Juice", 10, 0, notes= 'Fresh')
upsert_recipe_line("Green Dream", "Matcha Syrup", 5, 1)
upsert_recipe_line("Green Dream", "Miraculous Foam", 0, 0)

# --- Garden Negroni ---
upsert_recipe_line("Garden Negroni", "Plymouth Gin", 25, 1)
upsert_recipe_line("Garden Negroni", "Yellow Chartreuse", 10, 1)
upsert_recipe_line("Garden Negroni", "Matcha Syrup", 5, 1)
upsert_recipe_line("Garden Negroni", "Mancino Sakura Vermouth", 20, 1)

# --- Churchill Manhattan ---
upsert_recipe_line("Churchill Manhattan", "Chivas 12y", 35, 1)
upsert_recipe_line("Churchill Manhattan", "Cacao Blanc Briottet", 10, 1)
upsert_recipe_line("Churchill Manhattan", "Crème de Banana", 10, 1)
upsert_recipe_line("Churchill Manhattan", "Karminia Barolo Chinato", 10, 1)

# --- George Martini ---
upsert_recipe_line("George Martini", "Grey Goose Vodka", 30, 1)
upsert_recipe_line("George Martini", "Absolut Citron", 20, 1)
upsert_recipe_line("George Martini", "Limoncello", 20, 1)

# --- Ohara Punch ---
upsert_recipe_line("Ohara Punch", "Havana 7", 50, 1)
upsert_recipe_line("Ohara Punch", "Vanilla Syrup", 7, 1)
upsert_recipe_line("Ohara Punch", "Passion Fruit Puree", 18, 1)
upsert_recipe_line("Ohara Punch", "Rum Caviar", 0, 0, notes= 'Garnish')

# --- Deep Sea Cosmo ---
upsert_recipe_line("Deep Sea Cosmo", "Absolut Elyx", 50, 1)
upsert_recipe_line("Deep Sea Cosmo", "Yuzu Orange", 25, 1)
upsert_recipe_line("Deep Sea Cosmo", "Ayala Champagne", 20, 0, notes= 'Top')

# --- Mr Sweeney ---
upsert_recipe_line("Mr Sweeney", "Absolut Elyx", 50, 1)
upsert_recipe_line("Mr Sweeney", "Dom Benedictine", 10, 1)
upsert_recipe_line("Mr Sweeney", "Lemon Juice", 10, 0)
upsert_recipe_line("Mr Sweeney", "Maraschino Cherries", 0, 0)
upsert_recipe_line("Mr Sweeney", "Miraculous Foam", 0, 0)

# --- Hot Stuff ---
upsert_recipe_line("Hot Stuff", "Del Maguey Vida Mezcal", 50, 1)
upsert_recipe_line("Hot Stuff", "Ancho Reyes", 25, 1)
upsert_recipe_line("Hot Stuff", "Vanilla Syrup", 25, 1)
upsert_recipe_line("Hot Stuff", "Lime Juice", 20, 0, notes= 'Fresh')
upsert_recipe_line("Hot Stuff", "Champagne Foam", 0, 0, notes = "Garnish, Waste Champagne")

# --- Negroni ---
upsert_recipe_line("Negroni", "Plymouth Gin", 50, 1)
upsert_recipe_line("Negroni", "Campari", 25, 1)
upsert_recipe_line("Negroni", "Mancino Rosso", 20, 0, notes= 'Top')

# Cool Calm Cucumber
upsert_recipe_line("Cool Calm Cucumber", "Monkey 47", 40, 1)
upsert_recipe_line("Cool Calm Cucumber", "Midori", 15, 1)
upsert_recipe_line("Cool Calm Cucumber", "Lychee Liqueur", 5, 1)
upsert_recipe_line("Cool Calm Cucumber", "Lemon Juice", 10, 0, notes="fresh")
upsert_recipe_line("Cool Calm Cucumber", "Cucumber Sherbet", 5, 1)
upsert_recipe_line("Cool Calm Cucumber", "Miraculous Foam", 0, 0)
upsert_recipe_line("Cool Calm Cucumber", "Garden Mint", 1, 1)

# Garden of Temptation
upsert_recipe_line("Garden of Temptation", "Grey Goose Vodka", 40, 1)
upsert_recipe_line("Garden of Temptation", "Apple Juice", 20, 0, notes="fresh")
upsert_recipe_line("Garden of Temptation", "Manzana Verde Liqueur", 10, 1)
upsert_recipe_line("Garden of Temptation", "Lemon Juice", 10, 0, notes="fresh")
upsert_recipe_line("Garden of Temptation", "Miraculous Foam", 0, 0)
upsert_recipe_line("Garden of Temptation", "Matcha Syrup", 5, 1)

# Daylight Robbery
upsert_recipe_line("Daylight Robbery", "Altos Plata", 30, 1)
upsert_recipe_line("Daylight Robbery", "Del Maguey Vida Mezcal", 15, 1)
upsert_recipe_line("Daylight Robbery", "Mancino Sakura Vermouth", 20, 1)
upsert_recipe_line("Daylight Robbery", "Cointreau", 15, 1)
upsert_recipe_line("Daylight Robbery", "Lemon Juice", 20, 0, notes="fresh")
upsert_recipe_line("Daylight Robbery", "Sugar Syrup", 5, 1)
upsert_recipe_line("Daylight Robbery", "Absinthe Wash", 0, 0)

# The Devil's Tea
upsert_recipe_line("The Devil's Tea", "Del Maguey Vida Mezcal", 30, 1)
upsert_recipe_line("The Devil's Tea", "La Tomato", 10, 1)
upsert_recipe_line("The Devil's Tea", "Humami Vermouth", 10, 1)
upsert_recipe_line("The Devil's Tea", "Cocchi Rosa", 10, 1)
upsert_recipe_line("The Devil's Tea", "Wine Sherbet", 0, 0, notes="float")

# Pink Flamingo
upsert_recipe_line("Pink Flamingo", "Italicus", 25, 1)
upsert_recipe_line("Pink Flamingo", "Chambord", 15, 1)
upsert_recipe_line("Pink Flamingo", "Figue Liqueur", 5, 1)
upsert_recipe_line("Pink Flamingo", "Jasmin Green Pearls Cordial", 15, 1)
upsert_recipe_line("Pink Flamingo", "Ayala Champagne", 20, 0, notes="top")

# The Smooch
upsert_recipe_line("The Smooch", "La Hechicera Reserva", 40, 1)
upsert_recipe_line("The Smooch", "Kahlua", 10, 1)
upsert_recipe_line("The Smooch", "Dark Cacao Liqueur", 10, 1)
upsert_recipe_line("The Smooch", "Baileys", 5, 1)
upsert_recipe_line("The Smooch", "Sugar Syrup", 5, 1)

# Ship's Rumbullion
upsert_recipe_line("Ship's Rumbullion", "Havana 7", 20, 1)
upsert_recipe_line("Ship's Rumbullion", "Cointreau", 15, 1)
upsert_recipe_line("Ship's Rumbullion", "Tia Maria", 15, 1)
upsert_recipe_line("Ship's Rumbullion", "Disaronno", 15, 1)
upsert_recipe_line("Ship's Rumbullion", "Lime Juice", 20, 0, notes="fresh")
upsert_recipe_line("Ship's Rumbullion", "Sherry Syrup", 20, 1)

# TYYM
upsert_recipe_line("TYYM", "Goring Gin", 25, 1)
upsert_recipe_line("TYYM", "Lemon Juice", 10, 0, notes="fresh")
upsert_recipe_line("TYYM", "St Germain", 5, 1)
upsert_recipe_line("TYYM", "Crème de Peche", 5, 1)
upsert_recipe_line("TYYM", "Sherbet", 10, 1)
upsert_recipe_line("TYYM", "Rose Wine Syrup", 10, 1)
upsert_recipe_line("TYYM", "Ayala Champagne", 20, 0, notes="top")

# Mr Sweeney

upsert_recipe_line("Mr Sweeney", "Absolut Elyx", 50, 1)
upsert_recipe_line("Mr Sweeney", "Dom Benedictine", 10, 1)
upsert_recipe_line("Mr Sweeney", "Lemon Juice", 10, 0, notes="fresh")
upsert_recipe_line("Mr Sweeney", "Maraschino Cherries", 0, 0)
upsert_recipe_line("Mr Sweeney", "Miraculous Foam", 0, 0)

print("✅ Missing recipe lines inserted / updated")
con.commit()
pd.read_sql("""
SELECT c.cocktail_name, i.ingredient_name, b.ml_per_serve, b.include_in_cost, b.notes
FROM bridge_recipe b
JOIN dim_cocktail c ON b.cocktail_id = c.cocktail_id
JOIN dim_ingredient i ON b.ingredient_id = i.ingredient_id
ORDER BY c.cocktail_name, i.ingredient_name;
""", con)

✅ Missing recipe lines inserted / updated


,cocktail_name,ingredient_name,ml_per_serve,include_in_cost,notes
0,BLT,Absinthe Wash,0.0,0,NaN
1,BLT,Crème de Banana,5.0,1,NaN
2,BLT,Homemade Sweetcorn Syrup,5.0,1,NaN
3,BLT,Mancino Dry Vermouth,10.0,1,NaN
4,BLT,Whistle Pig,50.0,1,NaN
...,...,...,...,...,...
113,Tipsy Koala,Eucalyptus Syrup,5.0,1,NaN
114,Tipsy Koala,Garden Mint,1.0,1,NaN
115,Tipsy Koala,Grapefruit Juice,25.0,0,fresh
116,Tipsy Koala,Italicus,10.0,1,NaN


In [11]:
import sqlite3
import pandas as pd

# --- canonicalize dim_cocktail + repoint all tables that have cocktail_id ---
def canonicalize_dim_cocktail(DB_PATH: str):
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()

        # 1) Build canonical mapping: keep MIN(cocktail_id) per cocktail_name
        cur.executescript("""
        DROP TABLE IF EXISTS _cocktail_canonical_map;
        CREATE TEMP TABLE _cocktail_canonical_map AS
        SELECT
            cocktail_name,
            MIN(cocktail_id) AS canonical_id
        FROM dim_cocktail
        GROUP BY cocktail_name;

        DROP TABLE IF EXISTS _cocktail_id_remap;
        CREATE TEMP TABLE _cocktail_id_remap AS
        SELECT
            d.cocktail_id   AS old_id,
            m.canonical_id  AS new_id
        FROM dim_cocktail d
        JOIN _cocktail_canonical_map m
          ON m.cocktail_name = d.cocktail_name
        WHERE d.cocktail_id <> m.canonical_id;
        """)

        # Quick exit if nothing to fix
        n = cur.execute("SELECT COUNT(*) FROM _cocktail_id_remap;").fetchone()[0]
        if n == 0:
            print("✅ No duplicate cocktail_ids found in dim_cocktail.")
        else:
            print(f"⚠️ Found {n} duplicate cocktail_id rows to remap.")

        # 2) Re-point any table that has a cocktail_id column
        tables = [r[0] for r in cur.execute("""
            SELECT name FROM sqlite_master
            WHERE type='table' AND name NOT LIKE 'sqlite_%';
        """).fetchall()]

        def has_col(table: str, col: str) -> bool:
            cols = [r[1] for r in cur.execute(f"PRAGMA table_info('{table}');").fetchall()]
            return col in cols

        updated_tables = []
        for t in tables:
            if has_col(t, "cocktail_id"):
                cur.executescript(f"""
                UPDATE {t}
                SET cocktail_id = (
                    SELECT new_id
                    FROM _cocktail_id_remap r
                    WHERE r.old_id = {t}.cocktail_id
                )
                WHERE cocktail_id IN (SELECT old_id FROM _cocktail_id_remap);
                """)
                updated_tables.append(t)

        # 3) Delete duplicate dim_cocktail rows (keep canonical)
        cur.executescript("""
        DELETE FROM dim_cocktail
        WHERE cocktail_id IN (SELECT old_id FROM _cocktail_id_remap);
        """)

        # 4) Prevent future duplicates
        # (SQLite uses indexes as constraints for ON CONFLICT)
        cur.executescript("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_cocktail_name
        ON dim_cocktail(cocktail_name);
        """)

        con.commit()

        print("✅ Updated tables:", ", ".join(updated_tables))
        # sanity: show any remaining duplicates
        dup = pd.read_sql("""
            SELECT cocktail_name, COUNT(*) AS n
            FROM dim_cocktail
            GROUP BY cocktail_name
            HAVING COUNT(*) > 1;
        """, con)
        if len(dup):
            print("❌ Still duplicates exist (should be empty):")
            print(dup)
        else:
            print("✅ dim_cocktail is now unique by cocktail_name.")

canonicalize_dim_cocktail(DB_PATH)

✅ No duplicate cocktail_ids found in dim_cocktail.
✅ Updated tables: fact_cocktail_cost, fact_monthly_forecast__bak, fact_monthly_forecast, fact_2026_event_forecast_8e, fact_2026_baseline_7d, fact_monthly_sales, fact_reality_forecast_14f, dim_cocktail, bridge_recipe, fact_cocktail_price
✅ dim_cocktail is now unique by cocktail_name.


In [12]:
import sqlite3

def upsert_cocktail_price(con, cocktail_name, sell_price):
    con.execute("""
        INSERT INTO fact_cocktail_price (cocktail_id, sell_price)
        SELECT cocktail_id, ?
        FROM dim_cocktail
        WHERE cocktail_name = ?
        ON CONFLICT(cocktail_id) DO UPDATE SET
            sell_price=excluded.sell_price;
    """, (sell_price, cocktail_name))


# ---- ENTER YOUR MENU PRICES HERE ----
cocktail_prices = [
    ("Ginger Martini", 21),
    ("Earl Grey Martini", 21),
    ("Tipsy Koala", 22),
    ("Celeste", 25),
    ("Cool Calm Cucumber", 23),
    ("Hot Stuff", 25),
    ("Garden of Temptation", 22),
    ("Daylight Robbery", 28),
    ("The Devil's Tea", 22),
    ("Pink Flamingo", 24),
    ("The Smooch", 22),
    ("Ship's Rumbullion", 21),
    ("BLT", 35),
    ("Ristretto Martini", 23),
    ("Floral", 15),
    ("Green Dream", 15),
    ("Fruity Old Devil", 15),
    ("Garden Negroni", 25),
    ("Churchill Manhattan", 23),
    ("TYYM", 25),
    ("George Martini", 23),
    ("Ohara Punch", 25),
    ("Deep Sea Cosmo", 25),
    ("French Martini", 24),
    ("Negroni", 22),
]

with sqlite3.connect(DB_PATH) as con:
    for name, price in cocktail_prices:
        upsert_cocktail_price(con, name, price)

print("✅ Cocktail sell prices loaded")

✅ Cocktail sell prices loaded


#### Quick count check (so you know it loaded)

In [13]:

pd.read_sql("""
    SELECT c.cocktail_name, COUNT(*) AS lines
    FROM bridge_recipe b
    JOIN dim_cocktail c ON b.cocktail_id = c.cocktail_id
    GROUP BY c.cocktail_name
    ORDER BY lines DESC, c.cocktail_name;
""", con)



,cocktail_name,lines
0,Cool Calm Cucumber,7
1,Daylight Robbery,7
2,TYYM,7
3,Celeste,6
4,Garden of Temptation,6
5,Green Dream,6
6,Ship's Rumbullion,6
7,Tipsy Koala,6
8,BLT,5
9,Floral Twist,5


## 5) Build Cost and Margin views

In [14]:
import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    # View = saved query (so you can reuse it everywhere)
    con.execute("DROP VIEW IF EXISTS v_cocktail_cost;")
    con.execute("""
        CREATE VIEW v_cocktail_cost AS
        SELECT
            c.cocktail_id,
            c.cocktail_name,
            ROUND(SUM(
                CASE 
                    WHEN b.include_in_cost = 1 THEN b.ml_per_serve * i.cost_per_ml
                    ELSE 0
                END
            ), 4) AS cost_per_cocktail,
            ROUND(SUM(
                CASE 
                    WHEN b.include_in_cost = 1 THEN b.ml_per_serve
                    ELSE 0
                END
            ), 1) AS costed_ml
        FROM bridge_recipe b
        JOIN dim_cocktail c ON c.cocktail_id = b.cocktail_id
        JOIN dim_ingredient i ON i.ingredient_id = b.ingredient_id
        GROUP BY c.cocktail_id, c.cocktail_name;
    """)
    con.commit()

    # Preview
    df_cost = pd.read_sql("""
        SELECT *
        FROM v_cocktail_cost
        ORDER BY cost_per_cocktail DESC, cocktail_name;
    """, con)

df_cost

,cocktail_id,cocktail_name,cost_per_cocktail,costed_ml
0,13,BLT,6.1286,70.0
1,5,Cool Calm Cucumber,4.5711,66.0
2,4,Celeste,4.4568,85.0
3,6,Hot Stuff,4.3875,100.0
4,14,Ristretto Martini,3.7026,77.0
5,2,Earl Grey Martini,3.0654,70.0
6,11,The Smooch,2.9614,70.0
7,18,Garden Negroni,2.6962,60.0
8,25,Negroni,2.5529,75.0
9,8,Daylight Robbery,2.5401,85.0


#### Margin View

In [15]:
import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP VIEW IF EXISTS v_cocktail_margin;")
    con.execute("""
        CREATE VIEW v_cocktail_margin AS
        SELECT
            c.cocktail_name,
            p.sell_price,
            cc.cost_per_cocktail,
            ROUND(p.sell_price - cc.cost_per_cocktail, 2) AS profit_per_cocktail,
            ROUND((p.sell_price - cc.cost_per_cocktail) / p.sell_price, 4) AS margin_pct
        FROM dim_cocktail c
        JOIN fact_cocktail_price p ON p.cocktail_id = c.cocktail_id
        JOIN v_cocktail_cost cc ON cc.cocktail_id = c.cocktail_id;
    """)
    con.commit()

    df_margin = pd.read_sql("""
        SELECT *
        FROM v_cocktail_margin
        ORDER BY margin_pct ASC, cocktail_name;
    """, con)

df_margin

,cocktail_name,sell_price,cost_per_cocktail,profit_per_cocktail,margin_pct
0,Cool Calm Cucumber,23.0,4.5711,18.43,0.8013
1,Celeste,25.0,4.4568,20.54,0.8217
2,Hot Stuff,25.0,4.3875,20.61,0.8245
3,BLT,35.0,6.1286,28.87,0.8249
4,Ristretto Martini,23.0,3.7026,19.30,0.8390
5,Earl Grey Martini,21.0,3.0654,17.93,0.8540
6,The Smooch,22.0,2.9614,19.04,0.8654
7,Negroni,22.0,2.5529,19.45,0.8840
8,Garden of Temptation,22.0,2.5150,19.48,0.8857
9,Ship's Rumbullion,21.0,2.3168,18.68,0.8897


### 6) Historical Sales Fact Table

#### Load excel

In [16]:
import pandas as pd
import sqlite3

XLSX_PATH = "Monthly cocktail sales tracker 2024_2025 (2).xlsx"

# --- helpers ---
def normalise_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = s.replace("’", "'").replace("`", "'")
    while "  " in s:
        s = s.replace("  ", " ")
    return s

# --- 1) Load sheets ---
sheets = {
    "December 2024": (2024, 12),
    "January 2025":  (2025, 1),
    "February 2025": (2025, 2),
    "March 2025":    (2025, 3),
}

frames = []
for sheet, (year, month) in sheets.items():
    df = pd.read_excel(XLSX_PATH, sheet_name=sheet)
    df = df.rename(columns={"Cocktail": "cocktail_name", "Quantity sold": "quantity_sold"})
    df = df[["cocktail_name", "quantity_sold"]].copy()
    df["year"] = year
    df["month"] = month
    frames.append(df)

sales_raw = pd.concat(frames, ignore_index=True)

# --- 2) Clean + drop blanks ---
sales_raw["cocktail_name"] = sales_raw["cocktail_name"].map(normalise_name)
sales_raw["quantity_sold"] = pd.to_numeric(sales_raw["quantity_sold"], errors="coerce")

sales_raw = sales_raw[
    sales_raw["cocktail_name"].notna()
    & (sales_raw["cocktail_name"].str.strip() != "")
    & (sales_raw["cocktail_name"].str.lower() != "nan")
].copy()

# --- 3) Build April synthetic (avg of Jan–Mar) ---
jan_mar = sales_raw[(sales_raw["year"] == 2025) & (sales_raw["month"].isin([1,2,3]))].copy()

april = jan_mar.groupby("cocktail_name", as_index=False)["quantity_sold"].mean()
april["quantity_sold"] = april["quantity_sold"].round()
april["year"] = 2025
april["month"] = 4

sales_backfill = pd.concat([sales_raw, april], ignore_index=True)

# --- 4) Apply your alias mapping ---
name_map = {
    "Gin Martini": "Ginger Martini",
    "Vodka Martini": "Earl Grey Martini",
    "TYYMajesty": "TYYM",
    "C.C. Cucumber": "Cool Calm Cucumber",
    "Ships Rumbillion": "Ship's Rumbullion",
    "O'hara punch": "Ohara Punch",
    "Typsy koala": "Tipsy Koala",
    "Mr Green": "Green Dream",
    "The devils tea": "The Devil's Tea",

    # FIX: dim has Floral Twist (no "(NA)")
    "Floral": "Floral Twist",
    "Floral Twist (NA)": "Floral Twist",
    # Shorthand -> dim names (based on your dim_cocktail list)
    "Lock sock": "The Smooch",
    "Mirror": "Daylight Robbery",
    "Sazerac": "Churchill Manhattan",
}

sales_backfill["cocktail_name"] = sales_backfill["cocktail_name"].replace(name_map)
sales_backfill["cocktail_name"] = sales_backfill["cocktail_name"].map(normalise_name)

# collapse duplicates per month/cocktail
sales_backfill["quantity_sold"] = pd.to_numeric(sales_backfill["quantity_sold"], errors="coerce").fillna(0)
sales_backfill = sales_backfill.groupby(["year","month","cocktail_name"], as_index=False)["quantity_sold"].sum()

# --- 5) Map to dim_cocktail ---
with sqlite3.connect(DB_PATH) as con:
    dim = pd.read_sql("SELECT cocktail_id, cocktail_name FROM dim_cocktail;", con)

dim["cocktail_name"] = dim["cocktail_name"].map(normalise_name)
dim["key"] = dim["cocktail_name"].str.lower()

sales_backfill["key"] = sales_backfill["cocktail_name"].str.lower()

mapped = sales_backfill.merge(dim[["cocktail_id","cocktail_name","key"]], on="key", how="left", suffixes=("_excel","_dim"))

# report any remaining unmatched
missing = mapped[mapped["cocktail_id"].isna()][["year","month","cocktail_name_excel"]].drop_duplicates()

print("✅ Created: sales_raw, sales_backfill, mapped")
print("Unmatched rows:", len(missing))
missing.head(50)

✅ Created: sales_raw, sales_backfill, mapped
Unmatched rows: 0


,year,month,cocktail_name_excel


In [17]:
import sqlite3

upsert_df = mapped.dropna(subset=["cocktail_id"]).copy()
upsert_df["cocktail_id"] = upsert_df["cocktail_id"].astype(int)
upsert_df["quantity_sold"] = upsert_df["quantity_sold"].astype(int)

rows = list(
    upsert_df[["cocktail_id","year","month","quantity_sold"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()
    cur.executemany("""
        INSERT INTO fact_monthly_sales (cocktail_id, year, month, quantity_sold)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(cocktail_id, year, month)
        DO UPDATE SET quantity_sold = excluded.quantity_sold;
    """, rows)
    con.commit()

print(f"✅ Upserted {len(rows)} rows into fact_monthly_sales")

✅ Upserted 153 rows into fact_monthly_sales


#### 6B) — Create fact_monthly_sales

In [18]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS fact_monthly_sales (
            cocktail_id INTEGER NOT NULL,
            year INTEGER NOT NULL,
            month INTEGER NOT NULL,
            quantity_sold INTEGER NOT NULL,
            PRIMARY KEY (cocktail_id, year, month),
            FOREIGN KEY (cocktail_id) REFERENCES dim_cocktail(cocktail_id)
        );
    """)

print("✅ fact_monthly_sales ensured (no wipe)")

✅ fact_monthly_sales ensured (no wipe)


#### 6) — Add Financial Layer

In [19]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP VIEW IF EXISTS v_reality_financial_14g;")

    con.execute("""
    CREATE VIEW v_reality_financial_14g AS
    SELECT
        f.cocktail_id,
        f.year,
        f.month,

        f.reality_forecast_qty,

        p.price_per_unit,

        c.cost_per_unit,

        ROUND(f.reality_forecast_qty * p.price_per_unit,2) AS forecast_revenue,
        ROUND(f.reality_forecast_qty * c.cost_per_unit,2) AS forecast_cogs,
        ROUND(
            (f.reality_forecast_qty * p.price_per_unit)
            -
            (f.reality_forecast_qty * c.cost_per_unit)
        ,2) AS forecast_gross_profit

    FROM fact_reality_forecast_14f f

    LEFT JOIN fact_cocktail_price p
        ON p.cocktail_id = f.cocktail_id

    LEFT JOIN v_cost_per_cocktail c
        ON c.cocktail_id = f.cocktail_id
    """)

## 7) Executive Engine Outputs (Timeline + Drivers + Reality Forecast)

This section creates the continuous monthly timeline (Dec-2024 → Nov-2026),
applies non-compounding event drivers, and produces the financial engine views
used by the executive notebook.

#### 14A) Build 24-Month Timeline Scaffold

In [20]:
import pandas as pd
import sqlite3

START_Y, START_M = 2024, 12
MONTHS_TOTAL = 24

start = pd.Timestamp(year=START_Y, month=START_M, day=1)
months = pd.date_range(start=start, periods=MONTHS_TOTAL, freq="MS")

df_months = pd.DataFrame({
    "year": months.year.astype(int),
    "month": months.month.astype(int)
})

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    # Month dimension
    cur.execute("DROP TABLE IF EXISTS dim_month_14a;")
    cur.execute("""
        CREATE TABLE dim_month_14a (
            year  INTEGER NOT NULL,
            month INTEGER NOT NULL,
            PRIMARY KEY (year, month)
        );
    """)
    df_months.to_sql("dim_month_14a", con, if_exists="append", index=False)

    # Adjust sales (NO doubling)
    cur.execute("DROP VIEW IF EXISTS v_monthly_sales_adjusted_14a;")
    cur.execute("""
        CREATE VIEW v_monthly_sales_adjusted_14a AS
        SELECT
            cocktail_id,
            year,
            month,
            quantity_sold AS quantity_sold_adjusted
        FROM fact_monthly_sales;
    """)

    # Scaffold (all cocktails × all months)
    cur.execute("DROP VIEW IF EXISTS v_sales_14a_scaffold;")
    cur.execute("""
        CREATE VIEW v_sales_14a_scaffold AS
        SELECT
            d.cocktail_id,
            d.cocktail_name,
            m.year,
            m.month,
            COALESCE(a.quantity_sold_adjusted, 0) AS quantity_sold
        FROM dim_cocktail d
        CROSS JOIN dim_month_14a m
        LEFT JOIN v_monthly_sales_adjusted_14a a
          ON a.cocktail_id = d.cocktail_id
         AND a.year = m.year
         AND a.month = m.month;
    """)

    con.commit()

print("✅ 14A: Scaffold built (24 months) — NO December doubling")

✅ 14A: Scaffold built (24 months) — NO December doubling


In [21]:
import pandas as pd, sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT year, month, SUM(quantity_sold) AS total_qty
        FROM v_sales_14a_scaffold
        WHERE year=2024 AND month=12
        GROUP BY year, month;
    """, con)

df

,year,month,total_qty
0,2024,12,1794


#### 14B) Fill Missing Months

In [22]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_sales_14b_filled;")
    cur.execute("""
        CREATE VIEW v_sales_14b_filled AS
        WITH base AS (
            SELECT
                cocktail_id,
                cocktail_name,
                year,
                month,
                quantity_sold
            FROM v_sales_14a_scaffold
        ),
        filled AS (
            SELECT
                b1.cocktail_id,
                b1.cocktail_name,
                b1.year,
                b1.month,
                CASE
                    WHEN b1.quantity_sold > 0 THEN b1.quantity_sold
                    ELSE (
                        SELECT b2.quantity_sold
                        FROM base b2
                        WHERE b2.cocktail_id = b1.cocktail_id
                          AND (b2.year < b1.year OR (b2.year = b1.year AND b2.month < b1.month))
                          AND b2.quantity_sold > 0
                        ORDER BY b2.year DESC, b2.month DESC
                        LIMIT 1
                    )
                END AS quantity_sold_filled
            FROM base b1
        )
        SELECT
            cocktail_id,
            cocktail_name,
            year,
            month,
            COALESCE(quantity_sold_filled, 0) AS quantity_sold_filled
        FROM filled;
    """)

    con.commit()

print("✅ 14B: Carry-forward fill built")

✅ 14B: Carry-forward fill built


#### 14C) Add 3-Month Moving Average

In [23]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_sales_14c_features;")
    cur.execute("""
        CREATE VIEW v_sales_14c_features AS
        WITH ordered AS (
            SELECT
                cocktail_id,
                cocktail_name,
                year,
                month,
                quantity_sold_filled,
                (year * 12 + month) AS ym_key
            FROM v_sales_14b_filled
        )
        SELECT
            o.cocktail_id,
            o.cocktail_name,
            o.year,
            o.month,
            o.quantity_sold_filled,
            (o.ym_key - (SELECT MIN(ym_key) FROM ordered) + 1) AS month_index,
            ROUND((
                SELECT AVG(o2.quantity_sold_filled)
                FROM ordered o2
                WHERE o2.cocktail_id = o.cocktail_id
                  AND o2.ym_key BETWEEN o.ym_key - 2 AND o.ym_key
            ), 2) AS baseline_ma3
        FROM ordered o;
    """)

    con.commit()

print("✅ 14C: Features + MA3 baseline built")

✅ 14C: Features + MA3 baseline built


In [24]:
import pandas as pd, sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT year, MIN(month) AS min_m, MAX(month) AS max_m, COUNT(DISTINCT month) AS n_months
        FROM v_sales_14a_scaffold
        GROUP BY year
        ORDER BY year;
    """, con)

df

,year,min_m,max_m,n_months
0,2024,12,12,1
1,2025,1,12,12
2,2026,1,11,11


#### 14D-1) Demand drivers table (ROYAL + GARDEN + COCKTAIL)

In [25]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS fact_demand_drivers;")
    cur.execute("""
        CREATE TABLE fact_demand_drivers (
            driver_date TEXT NOT NULL,         -- YYYY-MM-01 marker is fine
            driver_type TEXT NOT NULL,         -- ROYAL | GARDEN | COCKTAIL
            target_scope TEXT NOT NULL,        -- ALL | CATEGORY | COCKTAIL
            target_value TEXT NOT NULL,        -- ALL, MARTINI, Negroni, etc.
            uplift REAL NOT NULL,              -- 0.20 = +20%
            notes TEXT,
            PRIMARY KEY (driver_date, driver_type, target_scope, target_value)
        );
    """)

    # ROYAL markers (month effect)
    cur.executemany("""
        INSERT INTO fact_demand_drivers
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (?, 'ROYAL', ?, ?, ?, ?)
        ON CONFLICT(driver_date, driver_type, target_scope, target_value)
        DO UPDATE SET uplift=excluded.uplift, notes=excluded.notes;
    """, [
        ("2025-02-01", "ALL", "ALL", 0.15, "Royal investiture month effect (Feb)"),
        ("2025-03-01", "ALL", "ALL", 0.15, "Royal investiture month effect (Mar)"),
        ("2025-06-01", "ALL", "ALL", 0.20, "Royal investiture month effect (Jun)"),
        ("2025-12-01", "ALL", "ALL", 0.15, "Royal December placeholder (month effect)"),
    ])

    # GARDEN FUNCTIONS (month markers)
    cur.executemany("""
        INSERT INTO fact_demand_drivers
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (?, 'GARDEN', ?, ?, ?, ?)
        ON CONFLICT(driver_date, driver_type, target_scope, target_value)
        DO UPDATE SET uplift=excluded.uplift, notes=excluded.notes;
    """, [
        ("2025-05-01", "ALL", "ALL", 0.20, "Garden functions month effect (May)"),
        ("2025-07-01", "ALL", "ALL", 0.15, "Garden functions month effect (Jul)"),
    ])

    # COCKTAIL PROMO markers (optional, keep minimal & relevant)
    cur.executemany("""
        INSERT INTO fact_demand_drivers
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (?, 'COCKTAIL', ?, ?, ?, ?)
        ON CONFLICT(driver_date, driver_type, target_scope, target_value)
        DO UPDATE SET uplift=excluded.uplift, notes=excluded.notes;
    """, [
        ("2025-06-01", "COCKTAIL", "Negroni", 0.25, "Negroni Month marker (Jun)"),
        ("2025-03-01", "CATEGORY", "MARTINI", 0.10, "Martini category bump (Mar)"),
    ])

    con.commit()

print("✅ 14D-1: fact_demand_drivers created (month markers only)")

✅ 14D-1: fact_demand_drivers created (month markers only)


In [26]:
import sqlite3

summer_rows = [
    # year, month, driver_type, target_scope, target_value, uplift, notes
    (2025, 7,  "SEASON_SUMMER", "ALL", "ALL", 0.20, "Summer peak uplift"),
    (2025, 8,  "SEASON_SUMMER", "ALL", "ALL", 0.20, "Summer peak uplift"),
    (2025, 9,  "SEASON_SUMMER", "ALL", "ALL", 0.10, "Late summer shoulder uplift"),
]

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.executemany("""
        INSERT INTO fact_demand_drivers 
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (
            printf('%04d-%02d-01', ?, ?),
            ?, ?, ?, ?, ?
        );
    """, summer_rows)

    con.commit()

print("✅ Summer seasonal drivers inserted")

✅ Summer seasonal drivers inserted


In [27]:
import sqlite3

rows = [
    (2025, 6,  "SEASON_RAMP",     "ALL", "ALL", 0.05, "Start of high season ramp"),
    (2025, 10, "SEASON_RAMP",     "ALL", "ALL", 0.10, "Autumn uplift / steady high trading"),
    (2025, 11, "SEASON_CHRISTMAS","ALL", "ALL", 0.30, "Christmas parties begin"),
    (2025, 12, "SEASON_CHRISTMAS","ALL", "ALL", 0.40, "Christmas peak"),
]

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.executemany("""
        INSERT INTO fact_demand_drivers
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (
            printf('%04d-%02d-01', ?, ?),
            ?, ?, ?, ?, ?
        );
    """, rows)

    con.commit()

print("✅ Added 2025 high-season + Christmas ramp drivers")

✅ Added 2025 high-season + Christmas ramp drivers


In [28]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("""
        INSERT INTO fact_demand_drivers
            (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES
            ('2026-01-01', 'SEASON', 'ALL', 'ALL', -0.25, 'Dry January (hospitality dip)')
        ON CONFLICT(driver_date, driver_type, target_scope, target_value)
        DO UPDATE SET uplift=excluded.uplift, notes=excluded.notes;
    """)
    con.commit()

print("✅ Added Dry January driver: 2026-01 (-25%)")

✅ Added Dry January driver: 2026-01 (-25%)


In [29]:
import sqlite3

rows_2026 = [

    # Royal event 
    ("2026-02-01", 'ROYAL', "ALL", "ALL", 0.15, "Royal investiture month effect (Feb)"),
    ("2026-03-01", 'ROYAL', "ALL", "ALL", 0.15, "Royal investiture month effect (Mar)"),
    ("2026-06-01", 'ROYAL', "ALL", "ALL", 0.20, "Royal investiture month effect (Jun)"),
    ("2026-12-01", 'ROYAL', "ALL", "ALL", 0.15, "Royal December placeholder (month effect)"),

    # Seasonal core
    ("2026-01-01", "SEASON", "ALL", "ALL", -0.25, "Dry January(Hospitality Dip)"),
    
    # Summer / high season (you said June should be at least 20 too)
    ("2026-06-01", "SEASON_SUMMER", "ALL", "ALL", 0.20, "Summer uplift (Jun)"),
    ("2026-07-01", "SEASON_SUMMER", "ALL", "ALL", 0.20, "Summer uplift (Jul)"),
    ("2026-08-01", "SEASON_SUMMER", "ALL", "ALL", 0.20, "Summer uplift (Aug)"),
    ("2026-09-01", "SEASON_SUMMER", "ALL", "ALL", 0.10, "Late summer shoulder (Sep)"),

    # Autumn / Christmas ramp
    ("2026-10-01", "SEASON_RAMP", "ALL", "ALL", 0.10, "Autumn uplift / trading"),
    ("2026-11-01", "SEASON_CHRISTMAS", "ALL", "ALL", 0.30, "Christmas parties begin"),
    ("2026-12-01", "SEASON_CHRISTMAS", "ALL", "ALL", 0.40, "Christmas peak"),
]

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()
    cur.executemany("""
        INSERT INTO fact_demand_drivers
            (driver_date, driver_type, target_scope, target_value, uplift, notes)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(driver_date, driver_type, target_scope, target_value)
        DO UPDATE SET uplift=excluded.uplift, notes=excluded.notes;
    """, rows_2026)
    con.commit()

print("✅ Added 2026 seasonal + Christmas drivers")

✅ Added 2026 seasonal + Christmas drivers


#### 14D-2) Aggregate drivers to month

In [30]:
import sqlite3
import pandas as pd

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS fact_month_driver_uplift;")
    cur.execute("""
        CREATE TABLE fact_month_driver_uplift (
            year INTEGER NOT NULL,
            month INTEGER NOT NULL,
            total_uplift REAL NOT NULL,
            PRIMARY KEY (year, month)
        );
    """)

    cur.execute("""
        INSERT INTO fact_month_driver_uplift (year, month, total_uplift)
        SELECT
            CAST(substr(driver_date, 1, 4) AS INTEGER) AS year,
            CAST(substr(driver_date, 6, 2) AS INTEGER) AS month,
            ROUND(SUM(uplift), 4) AS total_uplift
        FROM fact_demand_drivers
        GROUP BY
            CAST(substr(driver_date, 1, 4) AS INTEGER),
            CAST(substr(driver_date, 6, 2) AS INTEGER);
    """)

    con.commit()

print("✅ 14D-2: Aggregated → fact_month_driver_uplift")

with sqlite3.connect(DB_PATH) as con:
    df_14d2 = pd.read_sql("""
        SELECT year, month, total_uplift
        FROM fact_month_driver_uplift
        ORDER BY year, month;
    """, con)

df_14d2

✅ 14D-2: Aggregated → fact_month_driver_uplift


,year,month,total_uplift
0,2025,2,0.15
1,2025,3,0.25
2,2025,5,0.20
3,2025,6,0.50
4,2025,7,0.35
5,2025,8,0.20
6,2025,9,0.10
7,2025,10,0.10
8,2025,11,0.30
9,2025,12,0.55


#### 14E) Apply monthly uplifts to your baseline

In [31]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_reality_forecast_14e;")
    cur.execute("""
        CREATE VIEW v_reality_forecast_14e AS
        SELECT
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,

            ROUND(f.ma3, 2) AS baseline_ma3,
            COALESCE(u.total_uplift, 0.0) AS uplift,

            CAST(ROUND(f.ma3 * (1.0 + COALESCE(u.total_uplift, 0.0)), 0) AS INTEGER) AS reality_forecast_qty

        FROM v_sales_14c_features f
        LEFT JOIN fact_month_driver_uplift u
          ON u.year = f.year
         AND u.month = f.month
        WHERE f.year = 2025
          AND f.month BETWEEN 5 AND 12;
    """)

    con.commit()

print("✅ 14E: v_reality_forecast_14e created (May–Dec 2025)")

✅ 14E: v_reality_forecast_14e created (May–Dec 2025)


In [32]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP VIEW IF EXISTS v_reality_forecast_14e;")
    con.execute("""
        CREATE VIEW v_reality_forecast_14e AS
        SELECT
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,
            f.baseline_ma3,
            COALESCE(u.total_uplift, 0) AS uplift,
            COALESCE(g.growth_factor, 1.0) AS growth_factor,
            ROUND(
                f.baseline_ma3
                * (1 + COALESCE(u.total_uplift, 0))
                * COALESCE(g.growth_factor, 1.0),
            2) AS reality_forecast_qty
        FROM v_sales_14c_features f
        LEFT JOIN fact_month_driver_uplift u
          ON u.year = f.year
         AND u.month = f.month
        LEFT JOIN fact_growth_factor_14d g
          ON g.year = f.year
         AND g.month = f.month
        ;
    """)
    con.commit()

print("✅ v_reality_forecast_14e rebuilt: baseline × (1+uplift) × growth_factor")

✅ v_reality_forecast_14e rebuilt: baseline × (1+uplift) × growth_factor


#### 14E-1: create the “reality forecast” view

In [33]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_reality_forecast_14e;")
    cur.execute("""
        CREATE VIEW v_reality_forecast_14e AS
        SELECT
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,
            f.baseline_ma3,
            COALESCE(u.total_uplift, 0) AS uplift,
            ROUND(
                f.baseline_ma3 * (1 + COALESCE(u.total_uplift, 0)),
                2
            ) AS reality_forecast_qty
        FROM v_sales_14c_features f
        LEFT JOIN fact_month_driver_uplift u
          ON u.year = f.year
         AND u.month = f.month;
    """)

    con.commit()

print("✅ 14E rebuilt using baseline_ma3")

✅ 14E rebuilt using baseline_ma3


In [34]:
import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    df_chk = pd.read_sql("""
        SELECT cocktail_name, year, month, baseline_ma3, uplift, reality_forecast_qty
        FROM v_reality_forecast_14e
        WHERE cocktail_name IN ('Negroni', 'French Martini', 'Fruity Old Devil', 'Ship’s Rumbullion', 'The Devil’s Tea')
        ORDER BY cocktail_name, year, month;
    """, con)

df_chk

,cocktail_name,year,month,baseline_ma3,uplift,reality_forecast_qty
0,French Martini,2024,12,22.0,0.00,22.0
1,French Martini,2025,1,19.0,0.00,19.0
2,French Martini,2025,2,18.0,0.15,20.7
3,French Martini,2025,3,28.0,0.25,35.0
4,French Martini,2025,4,32.0,0.00,32.0
...,...,...,...,...,...,...
115,The Devil’s Tea,2026,7,10.0,0.20,12.0
116,The Devil’s Tea,2026,8,10.0,0.20,12.0
117,The Devil’s Tea,2026,9,10.0,0.10,11.0
118,The Devil’s Tea,2026,10,10.0,0.10,11.0


### 14F) Persist reality forecast

In [35]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP TABLE IF EXISTS fact_reality_forecast_14f;")
    con.execute("""
        CREATE TABLE fact_reality_forecast_14f AS
        SELECT * 
        FROM v_reality_forecast_14e;
    """)
    con.commit()

print("✅ fact_reality_forecast_14f recreated from v_reality_forecast_14e")

✅ fact_reality_forecast_14f recreated from v_reality_forecast_14e


#### 14G) Reality Forecast

In [36]:
import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    print(pd.read_sql("""
        SELECT name, type
        FROM sqlite_master
        WHERE name IN ('v_reality_forecast_14e', 'fact_reality_forecast_14f', 'v_reality_financial_14g');
    """, con))

                        name   type
0    v_reality_financial_14g   view
1     v_reality_forecast_14e   view
2  fact_reality_forecast_14f  table


In [37]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP VIEW IF EXISTS v_reality_financial_14g;")
    con.execute("""
    CREATE VIEW v_reality_financial_14g AS
    SELECT
        f.cocktail_id,
        f.year,
        f.month,
        f.reality_forecast_qty,

        p.sell_price AS sell_price,
        c.cost_per_cocktail,

        ROUND(f.reality_forecast_qty * p.sell_price, 2) AS forecast_revenue,
        ROUND(f.reality_forecast_qty * c.cost_per_cocktail, 2) AS forecast_cogs,
        ROUND((f.reality_forecast_qty * p.sell_price) - (f.reality_forecast_qty * c.cost_per_cocktail), 2) AS forecast_gross_profit
    FROM fact_reality_forecast_14f f
    LEFT JOIN fact_cocktail_price p
      ON p.cocktail_id = f.cocktail_id
    LEFT JOIN v_cost_per_cocktail c
      ON c.cocktail_id = f.cocktail_id;
    """)
print("✅ v_reality_financial_14g rebuilt (sell_price + cost_per_cocktail)")

✅ v_reality_financial_14g rebuilt (sell_price + cost_per_cocktail)


#### 14H) — Monthly Summary

In [38]:
import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    df_14h = pd.read_sql("""
        SELECT
            year,
            month,
            SUM(reality_forecast_qty) AS total_qty,
            ROUND(SUM(forecast_revenue),2) AS total_revenue,
            ROUND(SUM(forecast_cogs),2) AS total_cogs,
            ROUND(SUM(forecast_gross_profit),2) AS total_gp
        FROM v_reality_financial_14g
        GROUP BY year, month
        ORDER BY year, month;
    """, con)

df_14h

,year,month,total_qty,total_revenue,total_cogs,total_gp
0,2024,12,1794.00,36350.00,3686.00,31454.17
1,2025,1,1342.50,27171.00,2730.26,23383.65
2,2025,2,1359.30,27749.18,2789.89,23813.87
3,2025,3,1183.32,24401.62,2370.04,20517.48
4,2025,4,967.01,20098.34,1946.89,16836.15
5,2025,5,1196.80,24787.60,2379.99,20691.96
6,2025,6,1428.00,29583.00,2890.29,24867.40
7,2025,7,1285.20,26624.70,2601.26,22380.66
8,2025,8,1142.40,23666.40,2312.21,19893.94
9,2025,9,1047.20,21694.20,2119.55,18236.10


In [39]:
import pandas as pd, sqlite3

with sqlite3.connect(DB_PATH) as con:
    chk = pd.read_sql("""
        SELECT 
            COUNT(*) AS rows,
            COALESCE(SUM(quantity_sold),0) AS qty_sum,
            MIN(year||'-'||printf('%02d',month)) AS min_month,
            MAX(year||'-'||printf('%02d',month)) AS max_month
        FROM fact_monthly_sales;
    """, con)

chk

,rows,qty_sum,min_month,max_month
0,138,5586,2024-12,2025-04


## 8) Archived Forecast Experiments (Not used in the final executive notebook)

These sections were exploratory models (scenario toggles, inflation, sensitivity).
They are retained for reference but are not part of the production forecast pipeline.

### STEP 11 — 10% Monthly Uplift Forecast (Dynamic View)

#### STEP 11A — Create Forecast View

with sqlite3.connect(DB_PATH) as con:

    con.execute("DROP VIEW IF EXISTS v_forecast_6m;")

    con.execute("""
        CREATE VIEW v_forecast_6m AS

        WITH latest_month AS (
            SELECT
                cocktail_id,
                MAX(year*100 + month) AS max_period
            FROM fact_monthly_sales
            GROUP BY cocktail_id
        ),

        latest_sales AS (
            SELECT
                f.cocktail_id,
                f.quantity_sold
            FROM fact_monthly_sales f
            JOIN latest_month lm
                ON f.cocktail_id = lm.cocktail_id
                AND (f.year*100 + f.month) = lm.max_period
        ),

        forecast_base AS (
            SELECT
                ls.cocktail_id,
                ls.quantity_sold,
                1 AS forecast_month
            FROM latest_sales ls

            UNION ALL

            SELECT
                cocktail_id,
                quantity_sold,
                forecast_month + 1
            FROM forecast_base
            WHERE forecast_month < 6
        )

        SELECT
            fb.cocktail_id,
            c.cocktail_name,
            fb.forecast_month,
            ROUND(fb.quantity_sold * POWER(1.10, fb.forecast_month), 0) AS forecast_qty,
            p.sell_price,
            cc.cost_per_cocktail,

            ROUND(
                fb.quantity_sold * POWER(1.10, fb.forecast_month) * p.sell_price,
                2
            ) AS forecast_revenue,

            ROUND(
                fb.quantity_sold * POWER(1.10, fb.forecast_month)
                * (p.sell_price - cc.cost_per_cocktail),
                2
            ) AS forecast_profit

        FROM forecast_base fb
        JOIN dim_cocktail c
            ON fb.cocktail_id = c.cocktail_id
        JOIN fact_cocktail_price p
            ON fb.cocktail_id = p.cocktail_id
        JOIN v_cocktail_cost cc
            ON fb.cocktail_id = cc.cocktail_id;
    """)



print("✅ v_forecast_6m created")

import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    deps = pd.read_sql("""
    SELECT name, sql
    FROM sqlite_master
    WHERE type IN ('view')
      AND sql IS NOT NULL
      AND (
        sql LIKE '%v_forecast_%'
        OR sql LIKE '%fact_%12%'
        OR sql LIKE '%fact_%13%'
        OR sql LIKE '%scenario%'
      )
    """, con)

deps[["name"]]

with sqlite3.connect(DB_PATH) as con:
    forecast = pd.read_sql("""
        SELECT *
        FROM v_forecast_6m
        ORDER BY cocktail_name, forecast_month;
    """, con)

forecast.head(20)

### STEP 12 — Scenario Toggle Table + 12-Month Compounding Forecast

#### STEP 12A — Create dim_scenario

with sqlite3.connect(DB_PATH) as con:

    con.executescript("""
    DROP TABLE IF EXISTS dim_scenario;

    CREATE TABLE dim_scenario (
        scenario_id INTEGER PRIMARY KEY AUTOINCREMENT,
        scenario_name TEXT UNIQUE NOT NULL,
        monthly_uplift REAL NOT NULL
    );
    """)

    # your default scenarios (edit/add any time)
    scenarios = [
        ("Low 5%", 0.05),
        ("Base 10%", 0.10),
        ("High 15%", 0.15),
    ]

    con.executemany("""
        INSERT INTO dim_scenario (scenario_name, monthly_uplift)
        VALUES (?, ?);
    """, scenarios)



print("✅ dim_scenario created + loaded")

#### 12B) Create forecast view v_forecast_12m_scenarios

with sqlite3.connect(DB_PATH) as con:

    con.execute("DROP VIEW IF EXISTS v_forecast_12m_scenarios;")

    con.execute("""
    CREATE VIEW v_forecast_12m_scenarios AS

    WITH latest_month AS (
        SELECT
            cocktail_id,
            MAX(year*100 + month) AS max_period
        FROM fact_monthly_sales
        GROUP BY cocktail_id
    ),

    latest_sales AS (
        SELECT
            f.cocktail_id,
            f.quantity_sold AS base_qty
        FROM fact_monthly_sales f
        JOIN latest_month lm
            ON f.cocktail_id = lm.cocktail_id
            AND (f.year*100 + f.month) = lm.max_period
    ),

    months AS (
        SELECT 1 AS forecast_month
        UNION ALL
        SELECT forecast_month + 1
        FROM months
        WHERE forecast_month < 12
    )

    SELECT
        s.scenario_name,
        s.monthly_uplift,
        m.forecast_month,

        c.cocktail_id,
        c.cocktail_name,

        ls.base_qty,

        ROUND(ls.base_qty * POWER(1.0 + s.monthly_uplift, m.forecast_month), 0) AS forecast_qty,

        p.sell_price,
        cc.cost_per_cocktail,

        ROUND(
            (ls.base_qty * POWER(1.0 + s.monthly_uplift, m.forecast_month)) * p.sell_price,
            2
        ) AS forecast_revenue,

        ROUND(
            (ls.base_qty * POWER(1.0 + s.monthly_uplift, m.forecast_month))
            * (p.sell_price - cc.cost_per_cocktail),
            2
        ) AS forecast_profit

    FROM dim_scenario s
    CROSS JOIN months m
    JOIN latest_sales ls
        ON 1=1
    JOIN dim_cocktail c
        ON c.cocktail_id = ls.cocktail_id
    JOIN fact_cocktail_price p
        ON p.cocktail_id = c.cocktail_id
    JOIN v_cocktail_cost cc
        ON cc.cocktail_id = c.cocktail_id;
    """)



print("✅ v_forecast_12m_scenarios created")

### 12C) — “Toggle” it

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT *
        FROM v_forecast_12m_scenarios
        WHERE scenario_name = 'Base 10%'
          AND cocktail_name = 'Negroni'
        ORDER BY forecast_month;
    """, con)

df.tail()
con.commit()

### 13) Forecast Targets + Month-End Actuals

#### 13A) Create forecast fact table

import sqlite3

with sqlite3.connect(DB_PATH) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS fact_monthly_forecast (
            cocktail_id   INTEGER NOT NULL,
            year          INTEGER NOT NULL,
            month         INTEGER NOT NULL,
            scenario_name TEXT    NOT NULL,
            forecast_qty  INTEGER NOT NULL,
            PRIMARY KEY (cocktail_id, year, month, scenario_name)
        );
    """)
    con.commit()

print("13A: fact_monthly_forecast ready (composite PK for UPSERT)")



#### 13B)Generate next-month forecast (+10% from latest actual)

YEAR_NEXT = 2025
MONTH_NEXT = 5
SCENARIO = "Baseline +10%"

with sqlite3.connect(DB_PATH) as con:
    base = pd.read_sql("""
        SELECT f.cocktail_id, f.quantity_sold
        FROM fact_monthly_sales f
        JOIN (
            SELECT cocktail_id, MAX(year*100 + month) AS max_period
            FROM fact_monthly_sales
            GROUP BY cocktail_id
        ) lm
        ON f.cocktail_id = lm.cocktail_id
        AND (f.year*100 + f.month) = lm.max_period;
    """, con)

base["forecast_qty"] = (base["quantity_sold"] * 1.10).round().astype(int)
base["year"] = YEAR_NEXT
base["month"] = MONTH_NEXT
base["scenario_name"] = SCENARIO

with sqlite3.connect(DB_PATH) as con:
    for row in base[["cocktail_id","year","month","scenario_name","forecast_qty"]].itertuples(index=False):
        con.execute("""
            INSERT INTO fact_monthly_forecast (cocktail_id, year, month, scenario_name, forecast_qty)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(cocktail_id, year, month, scenario_name)
            DO UPDATE SET forecast_qty = excluded.forecast_qty;
        """, tuple(row))
    con.commit()

print("13B: Baseline forecast saved")

#### 13C) Month-end actuals

import sqlite3

YEAR = 2025
MONTH = 4
QTY = 210
COCKTAIL_NAME = "Negroni"

with sqlite3.connect(DB_PATH) as con:
    row = con.execute("""
        SELECT cocktail_id
        FROM dim_cocktail
        WHERE cocktail_name = ?
        LIMIT 1;
    """, (COCKTAIL_NAME,)).fetchone()

    if not row:
        raise ValueError(f"❌ Cocktail not found in dim_cocktail: {COCKTAIL_NAME}")

    cocktail_id = row[0]

    con.execute("""
        INSERT INTO fact_monthly_sales (cocktail_id, year, month, quantity_sold)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(cocktail_id, year, month) DO UPDATE SET
            quantity_sold = excluded.quantity_sold;
    """, (cocktail_id, YEAR, MONTH, QTY))

    con.commit()

print(f"✅ 13C: Actual saved — {COCKTAIL_NAME} ({cocktail_id}) {YEAR}-{MONTH:02d} qty={QTY}")

#### 13D) View: forecast vs actual financial

import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    # Stable alias expected by the financial view
    cur.execute("DROP VIEW IF EXISTS v_cost_per_cocktail;")
    cur.execute("""
        CREATE VIEW v_cost_per_cocktail AS
        SELECT cocktail_id, cost_per_cocktail
        FROM v_cocktail_cost;
    """)

    cur.execute("DROP VIEW IF EXISTS v_forecast_vs_actual_financial;")
    cur.execute("""
        CREATE VIEW v_forecast_vs_actual_financial AS
        SELECT
            f.cocktail_id,
            f.year,
            f.month,
            f.scenario_name,

            f.forecast_qty,
            COALESCE(a.quantity_sold, 0) AS actual_qty,

            (COALESCE(a.quantity_sold, 0) - f.forecast_qty) AS variance_qty,

            ROUND(
                100.0 * (COALESCE(a.quantity_sold, 0) - f.forecast_qty)
                / NULLIF(f.forecast_qty, 0),
                2
            ) AS variance_pct,

            ROUND(f.forecast_qty * p.sell_price, 2) AS forecast_revenue,
            ROUND(COALESCE(a.quantity_sold, 0) * p.sell_price, 2) AS actual_revenue,

            ROUND(f.forecast_qty * c.cost_per_cocktail, 2) AS forecast_cogs,
            ROUND(COALESCE(a.quantity_sold, 0) * c.cost_per_cocktail, 2) AS actual_cogs,

            ROUND((f.forecast_qty * p.sell_price) - (f.forecast_qty * c.cost_per_cocktail), 2) AS forecast_gross_profit,
            ROUND((COALESCE(a.quantity_sold, 0) * p.sell_price) - (COALESCE(a.quantity_sold, 0) * c.cost_per_cocktail), 2) AS actual_gross_profit,

            ROUND(
              ((COALESCE(a.quantity_sold, 0) * p.sell_price) - (COALESCE(a.quantity_sold, 0) * c.cost_per_cocktail))
              -
              ((f.forecast_qty * p.sell_price) - (f.forecast_qty * c.cost_per_cocktail)),
              2
            ) AS variance_gross_profit

        FROM fact_monthly_forecast f
        LEFT JOIN fact_monthly_sales a
          ON a.cocktail_id = f.cocktail_id
         AND a.year = f.year
         AND a.month = f.month
        LEFT JOIN fact_cocktail_price p
          ON p.cocktail_id = f.cocktail_id
        LEFT JOIN v_cost_per_cocktail c
          ON c.cocktail_id = f.cocktail_id;
    """)

    con.commit()

print("✅ 13D: v_forecast_vs_actual_financial rebuilt")

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT *
        FROM v_forecast_vs_actual
        ORDER BY variance_pct DESC;
    """, con)

df

#### 13F) Event-Based Forecast Generator


import pandas as pd
import sqlite3

# 13F SETTINGS (Non-Compounding)

YEAR_NEXT = 2025
MONTH_NEXT = 5

SCENARIO = "Chelsea Flower Show"
UPLIFT = 0.20  # single uplift, NOT compounding

# -----------------------------
# 1) Baseline = latest actual month per cocktail
# -----------------------------
with sqlite3.connect(DB_PATH) as con:
    base = pd.read_sql("""
        SELECT f.cocktail_id, f.quantity_sold
        FROM fact_monthly_sales f
        JOIN (
            SELECT cocktail_id, MAX(year*100 + month) AS max_period
            FROM fact_monthly_sales
            GROUP BY cocktail_id
        ) lm
          ON f.cocktail_id = lm.cocktail_id
         AND (f.year*100 + f.month) = lm.max_period;
    """, con)

# -----------------------------
# 2) Apply SINGLE uplift (non-compounding)
# -----------------------------
base["forecast_qty"] = (base["quantity_sold"] * (1 + UPLIFT)).round().astype(int)
base["year"] = YEAR_NEXT
base["month"] = MONTH_NEXT
base["scenario_name"] = SCENARIO

# -----------------------------
# 3) UPSERT into fact_monthly_forecast
#    (requires PRIMARY KEY on cocktail_id, year, month, scenario_name)
# -----------------------------
with sqlite3.connect(DB_PATH) as con:
    for row in base[["cocktail_id","year","month","scenario_name","forecast_qty"]].itertuples(index=False):
        con.execute("""
            INSERT INTO fact_monthly_forecast (cocktail_id, year, month, scenario_name, forecast_qty)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(cocktail_id, year, month, scenario_name)
            DO UPDATE SET forecast_qty = excluded.forecast_qty;
        """, tuple(row))
    con.commit()

print(f"✅ 13F saved: {SCENARIO} ({YEAR_NEXT}-{MONTH_NEXT:02d}), uplift={UPLIFT:.0%}, rows={len(base)}")

#### 13G) Scenario Toggle

import sqlite3

ACTIVE_SCENARIO = "Baseline +10%"

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS config_active_scenario (
            id INTEGER PRIMARY KEY CHECK (id = 1),
            scenario_name TEXT NOT NULL
        );
    """)

    cur.execute("""
        INSERT INTO config_active_scenario (id, scenario_name)
        VALUES (1, ?)
        ON CONFLICT(id) DO UPDATE SET scenario_name = excluded.scenario_name;
    """, (ACTIVE_SCENARIO,))

    cur.execute("DROP VIEW IF EXISTS v_active_forecast;")
    cur.execute("""
        CREATE VIEW v_active_forecast AS
        SELECT f.*
        FROM fact_monthly_forecast f
        JOIN config_active_scenario s
          ON f.scenario_name = s.scenario_name;
    """)

    con.commit()

print(f"✅ 13G: Active scenario set to '{ACTIVE_SCENARIO}'")

#### 13H) Financial performance by scenario summary 

import pandas as pd
import sqlite3

# -----------------------------
# 13H SETTINGS
# -----------------------------
YEAR_FOCUS = 2025
MONTH_FOCUS = 5
SCENARIO_FOCUS = "Chelsea Flower Show"

pd.set_option("display.max_columns", None)

with sqlite3.connect(DB_PATH) as con:
    df_13h = pd.read_sql("""
        SELECT
            scenario_name,
            year,
            month,

            SUM(forecast_qty) AS forecast_qty_total,
            SUM(actual_qty) AS actual_qty_total,

            ROUND(SUM(forecast_revenue), 2) AS forecast_revenue_total,
            ROUND(SUM(actual_revenue), 2) AS actual_revenue_total,

            ROUND(SUM(forecast_cogs), 2) AS forecast_cogs_total,
            ROUND(SUM(actual_cogs), 2) AS actual_cogs_total,

            ROUND(SUM(forecast_gross_profit), 2) AS forecast_gp_total,
            ROUND(SUM(actual_gross_profit), 2) AS actual_gp_total,

            ROUND(SUM(variance_gross_profit), 2) AS variance_gp_total
        FROM v_forecast_vs_actual_financial
        WHERE year = ?
          AND month = ?
          AND scenario_name = ?
        GROUP BY scenario_name, year, month;
    """, con, params=(YEAR_FOCUS, MONTH_FOCUS, SCENARIO_FOCUS))

print("13H loaded")
df_13h

import sqlite3

prices_to_insert = {
    9: 24.0,   # The Devil’s Tea
    12: 24.0,  # Ship’s Rumbullion
    15: 15.0,  # Floral Twist
    24: 22.0   # Mr Sweeney
}

with sqlite3.connect(DB_PATH) as con:
    for cid, price in prices_to_insert.items():
        con.execute("""
            INSERT INTO fact_cocktail_price (cocktail_id, sell_price)
            VALUES (?, ?)
            ON CONFLICT(cocktail_id)
            DO UPDATE SET sell_price = excluded.sell_price;
        """, (cid, price))
    con.commit()

print("✅ Missing sell_price values inserted/updated")

#### 13I) Active Scenario Financial

import pandas as pd
import sqlite3

YEAR_FOCUS = YEAR_NEXT
MONTH_FOCUS = MONTH_NEXT

with sqlite3.connect(DB_PATH) as con:
    df_active = pd.read_sql("""
        SELECT
            s.scenario_name AS active_scenario,
            f.year,
            f.month,

            SUM(f.forecast_qty) AS forecast_qty_total,
            SUM(f.actual_qty) AS actual_qty_total,

            ROUND(SUM(f.forecast_revenue), 2) AS forecast_revenue_total,
            ROUND(SUM(f.actual_revenue), 2) AS actual_revenue_total,

            ROUND(SUM(f.forecast_cogs), 2) AS forecast_cogs_total,
            ROUND(SUM(f.actual_cogs), 2) AS actual_cogs_total,

            ROUND(SUM(f.forecast_gross_profit), 2) AS forecast_gp_total,
            ROUND(SUM(f.actual_gross_profit), 2) AS actual_gp_total,

            ROUND(SUM(f.variance_gross_profit), 2) AS variance_gp_total
        FROM v_forecast_vs_actual_financial f
        JOIN config_active_scenario s
          ON f.scenario_name = s.scenario_name
        WHERE f.year = ?
          AND f.month = ?
        GROUP BY s.scenario_name, f.year, f.month;
    """, con, params=(YEAR_FOCUS, MONTH_FOCUS))

print("✅ 13I: Active scenario financial summary loaded")
df_active

#### 13J) Top Cocktails Driving Impact

import pandas as pd
import sqlite3

YEAR_FOCUS = 2025
MONTH_FOCUS = 5
SCENARIO_FOCUS = "Chelsea Flower Show"
TOP_N = 20

pd.set_option("display.max_columns", None)

with sqlite3.connect(DB_PATH) as con:
    df_13j = pd.read_sql("""
        SELECT
            f.scenario_name,
            f.year,
            f.month,
            f.cocktail_id,
            d.cocktail_name,

            f.forecast_qty,
            f.actual_qty,

            ROUND(f.forecast_revenue, 2) AS forecast_revenue,
            ROUND(f.forecast_cogs, 2) AS forecast_cogs,
            ROUND(f.forecast_gross_profit, 2) AS forecast_gross_profit,

            ROUND(f.actual_revenue, 2) AS actual_revenue,
            ROUND(f.actual_cogs, 2) AS actual_cogs,
            ROUND(f.actual_gross_profit, 2) AS actual_gross_profit,

            ROUND(f.variance_gross_profit, 2) AS variance_gross_profit
        FROM v_forecast_vs_actual_financial f
        JOIN dim_cocktail d
          ON d.cocktail_id = f.cocktail_id
        WHERE f.year = ?
          AND f.month = ?
          AND f.scenario_name = ?
        ORDER BY forecast_gross_profit DESC, forecast_qty DESC
        LIMIT ?;
    """, con, params=(YEAR_FOCUS, MONTH_FOCUS, SCENARIO_FOCUS, TOP_N))

print("13J loaded")
df_13j

import sqlite3

to_drop = [
    "fact_2026_baseline_7d",
    "fact_2026_event_forecast_8e",
    "fact_reality_forecast_14f",
]

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    existing = cur.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name IN ('fact_2026_baseline_7d','fact_2026_event_forecast_8e','fact_reality_forecast_14f');
    """).fetchall()
    existing = [r[0] for r in existing]

    for t in to_drop:
        if t in existing:
            cur.execute(f"DROP TABLE {t};")

    con.commit()

print("✅ Dropped derived tables that existed:", existing)

import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS fact_2026_event_forecast_8e;")
    cur.execute("DROP TABLE IF EXISTS fact_2026_baseline_7d;")
    cur.execute("DROP TABLE IF EXISTS fact_reality_forecast_14f;")

    con.commit()

print("✅ Derived forecast tables dropped — ready for clean rebuild")

#### 13K) Scenario Comparison (Baseline vs Chelsea)

import pandas as pd
import sqlite3

YEAR_FOCUS = 2025
MONTH_FOCUS = 5

SCENARIO_A = "Baseline +10%"
SCENARIO_B = "Chelsea Flower Show"

pd.set_option("display.max_columns", None)

with sqlite3.connect(DB_PATH) as con:
    df_13k = pd.read_sql("""
        SELECT
            d.cocktail_name,

            -- Baseline
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_qty END), 0) AS baseline_qty,
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_gross_profit END), 0) AS baseline_gp,

            -- Event
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_qty END), 0) AS event_qty,
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_gross_profit END), 0) AS event_gp,

            -- GP Difference
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_gross_profit END), 0)
            -
            COALESCE(SUM(CASE WHEN f.scenario_name = ? THEN f.forecast_gross_profit END), 0)
            AS gp_difference

        FROM dim_cocktail d
        LEFT JOIN v_forecast_vs_actual_financial f
          ON d.cocktail_id = f.cocktail_id
         AND f.year = ?
         AND f.month = ?
         AND f.scenario_name IN (?, ?)

        GROUP BY d.cocktail_name
        ORDER BY gp_difference DESC;
    """, con, params=(
        SCENARIO_A, SCENARIO_A,
        SCENARIO_B, SCENARIO_B,
        SCENARIO_B, SCENARIO_A,
        YEAR_FOCUS, MONTH_FOCUS,
        SCENARIO_A, SCENARIO_B
    ))

print("13K loaded (null-safe)")
df_13k

import pandas as pd, sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT cocktail_name
        FROM dim_cocktail
        WHERE cocktail_name LIKE '%’%';
    """, con)

df